# 04 Results and Diagnostics

This notebook evaluates the model trained on the new 25,000-simulation dataset. All checks use the held-out test set, which was not used during training or validation.

## Setup

In Colab, mount Google Drive before running this notebook. The model uses the PyTorch backend.

In [ ]:
from pathlib import Path
import os
import sys

os.environ["KERAS_BACKEND"] = "torch"

cwd = Path.cwd().resolve()
drive_root = Path("/content/drive/MyDrive/GW_Project")
local_root = cwd.parent if cwd.name == "notebooks" else cwd
PROJECT_ROOT = drive_root if (drive_root / "src").exists() else local_root
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Project folder not found. Mount Google Drive first when using Colab.")
sys.path.insert(0, str(PROJECT_ROOT))

DATASET_SIZE = 25000
DATASET_PATH = PROJECT_ROOT / "data" / f"gw_dataset_{DATASET_SIZE}.npz"
MODEL_DIR = PROJECT_ROOT / "models" / f"bayesflow_model_{DATASET_SIZE}"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
print("Project:", PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.model import PARAMETER_NAMES, load_npz_dataset, load_workflow, split_dataset_three_way

dataset = load_npz_dataset(DATASET_PATH)
train_data, validation_data, test_data = split_dataset_three_way(
    dataset, validation_fraction=0.1, test_fraction=0.1, seed=2026
)
workflow = load_workflow(MODEL_DIR)

print("Full dataset:", dataset["strain"].shape, dataset["parameters"].shape)
print("Training data:", train_data["strain"].shape)
print("Validation data:", validation_data["strain"].shape)
print("Test data:", test_data["strain"].shape)
print("Model:", MODEL_DIR / "model.keras")

## Draw posterior samples

We select test signals with a fixed random seed and draw posterior samples in batches. This same set is reused for every diagnostic.

In [ ]:
RNG = np.random.default_rng(2026)
N_DIAGNOSTICS = 200
N_POSTERIOR_SAMPLES = 500
INFERENCE_BATCH_SIZE = 25

indices = RNG.choice(len(test_data["parameters"]), size=N_DIAGNOSTICS, replace=False)
diagnostic_strain = test_data["strain"][indices]
true_parameters = test_data["parameters"][indices]

posterior_batches = []
for start in range(0, N_DIAGNOSTICS, INFERENCE_BATCH_SIZE):
    stop = min(start + INFERENCE_BATCH_SIZE, N_DIAGNOSTICS)
    posterior = workflow.sample(
        conditions={"strain": diagnostic_strain[start:stop]},
        num_samples=N_POSTERIOR_SAMPLES,
    )
    posterior_batches.append(np.asarray(posterior["parameters"]))

posterior_samples = np.concatenate(posterior_batches, axis=0)
assert posterior_samples.shape == (N_DIAGNOSTICS, N_POSTERIOR_SAMPLES, 6)
assert np.isfinite(posterior_samples).all()
print("Posterior samples:", posterior_samples.shape)

## Posterior example

The dashed line is the true parameter value for one test signal.

In [ ]:
example_samples = posterior_samples[0]
example_truth = true_parameters[0]

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    ax.hist(example_samples[:, i], bins=35, density=True, alpha=0.75)
    ax.axvline(example_truth[i], color="black", linestyle="--", label="true")
    ax.set_title(name)
    ax.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle("Posterior samples for one test signal")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "posterior_example.png", dpi=180, bbox_inches="tight")
plt.show()

## Parameter recovery

Each point compares a true value with the posterior mean. Good recovery places points close to the dashed diagonal.

In [ ]:
posterior_means = posterior_samples.mean(axis=1)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    truth = true_parameters[:, i]
    prediction = posterior_means[:, i]
    lower = min(truth.min(), prediction.min())
    upper = max(truth.max(), prediction.max())
    ax.scatter(truth, prediction, s=18, alpha=0.6)
    ax.plot([lower, upper], [lower, upper], "k--")
    ax.set(title=name, xlabel="True value", ylabel="Posterior mean")
    ax.grid(alpha=0.2)
fig.suptitle("Parameter recovery on held-out simulations")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "parameter_recovery.png", dpi=180, bbox_inches="tight")
plt.show()

## Simulation-based calibration

For a calibrated posterior, the rank of the true value among posterior samples should be uniform. We show both rank histograms and empirical CDFs. The shaded ECDF region is a 95% simultaneous band from the Dvoretzky–Kiefer–Wolfowitz bound.

In [ ]:
ranks = (posterior_samples < true_parameters[:, None, :]).sum(axis=1)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    ax.hist(ranks[:, i], bins=10, range=(0, N_POSTERIOR_SAMPLES), edgecolor="black")
    ax.set(title=name, xlabel="Rank", ylabel="Count")
    ax.grid(alpha=0.2)
fig.suptitle("SBC rank histograms")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sbc_simple.png", dpi=180, bbox_inches="tight")
plt.show()

rank_u = (ranks + 0.5) / (N_POSTERIOR_SAMPLES + 1.0)
grid = np.linspace(0.0, 1.0, 300)
epsilon = np.sqrt(np.log(2.0 / 0.05) / (2.0 * N_DIAGNOSTICS))
lower_band = np.maximum(0.0, grid - epsilon)
upper_band = np.minimum(1.0, grid + epsilon)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    sorted_ranks = np.sort(rank_u[:, i])
    ecdf = np.arange(1, N_DIAGNOSTICS + 1) / N_DIAGNOSTICS
    ax.fill_between(grid, lower_band, upper_band, color="lightgray", label="95% band")
    ax.plot(grid, grid, "k--", label="uniform")
    ax.step(sorted_ranks, ecdf, where="post", color="tab:blue")
    ax.set(title=name, xlabel="Normalized rank", ylabel="ECDF", xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle("SBC empirical CDFs")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sbc_ecdf.png", dpi=180, bbox_inches="tight")
plt.show()

# The same calibration result shown as ECDF minus the uniform reference.
# Pointwise binomial bands give the curved shape used in ECDF difference plots.
from scipy.stats import binom

difference_grid = np.linspace(0.0, 1.0, 500)
band_lower = binom.ppf(0.025, N_DIAGNOSTICS, difference_grid) / N_DIAGNOSTICS - difference_grid
band_upper = binom.ppf(0.975, N_DIAGNOSTICS, difference_grid) / N_DIAGNOSTICS - difference_grid

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    sorted_ranks = np.sort(rank_u[:, i])
    empirical_cdf = np.searchsorted(sorted_ranks, difference_grid, side="right") / N_DIAGNOSTICS
    ecdf_difference = empirical_cdf - difference_grid
    ax.fill_between(difference_grid, band_lower, band_upper, color="lightgray", label="95% confidence bands")
    ax.plot(difference_grid, ecdf_difference, color="midnightblue", label="Rank ECDF")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set(title=name, xlabel="Normalized rank statistic", ylabel="ECDF difference", xlim=(0, 1))
    ax.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle("SBC ECDF difference plots")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sbc_ecdf_difference.png", dpi=180, bbox_inches="tight")
plt.show()

## Posterior contraction

Posterior widths are divided by each parameter's prior width so different units can be compared. The dashed line is the standard deviation of a uniform prior after the same normalization. More distant signals usually have lower SNR and therefore wider posteriors.

In [ ]:
prior_widths = np.array([70.0, 70.0, 1.98, 1.98, 900.0, np.pi])
normalized_widths = posterior_samples.std(axis=1) / prior_widths
true_distance = true_parameters[:, 4]
prior_reference = 1.0 / np.sqrt(12.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(true_distance, normalized_widths[:, 4], alpha=0.6)
axes[0].set(title="Distance uncertainty", xlabel="True distance [Mpc]", ylabel="Posterior std / prior width")
axes[1].scatter(true_distance, normalized_widths[:, 5], alpha=0.6)
axes[1].set(title="Inclination uncertainty", xlabel="True distance [Mpc]", ylabel="Posterior std / prior width")
for ax in axes:
    ax.axhline(prior_reference, color="black", linestyle="--", label="prior std")
    ax.grid(alpha=0.2)
axes[0].legend()
fig.suptitle("Posterior contraction and source distance")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "posterior_contraction.png", dpi=180, bbox_inches="tight")
plt.show()

## Why inclination is difficult to recover

For a non-precessing binary, the two polarizations have approximate inclination factors

- $h_+ \propto (1 + \cos^2 \iota)/(2D)$
- $h_\times \propto \cos \iota/D$

and a single detector observes $h = F_+h_+ + F_\times h_\times$. This creates a strong distance–inclination degeneracy: changing distance can compensate for changing inclination. Face-on and face-off systems are also very similar, and the signal changes only weakly with inclination near those orientations. Therefore a broad or sometimes symmetric inclination posterior is expected, especially for distant low-SNR signals. This is a limitation of information in a single detector, not automatically a training failure. Calibration is more important than forcing a sharp point estimate.

In [ ]:
true_inclination = true_parameters[:, 5]
mean_inclination = posterior_means[:, 5]
mean_cos_inclination = np.cos(posterior_samples[:, :, 5]).mean(axis=1)
true_cos_inclination = np.cos(true_inclination)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(true_inclination, mean_inclination, c=true_distance, cmap="viridis", alpha=0.7)
axes[0].plot([0, np.pi], [0, np.pi], "k--")
axes[0].set(xlabel="True inclination [rad]", ylabel="Posterior mean [rad]", title="Inclination recovery")
scatter = axes[1].scatter(true_cos_inclination, mean_cos_inclination, c=true_distance, cmap="viridis", alpha=0.7)
axes[1].plot([-1, 1], [-1, 1], "k--")
axes[1].set(xlabel="True cos(inclination)", ylabel="Posterior mean cos(inclination)", title="Recovery in the sampled prior coordinate")
for ax in axes:
    ax.grid(alpha=0.2)
fig.colorbar(scatter, ax=axes, label="Distance [Mpc]")
fig.savefig(FIGURE_DIR / "inclination_recovery.png", dpi=180, bbox_inches="tight")
plt.show()

inclination_mae = np.mean(np.abs(mean_inclination - true_inclination))
print(f"Inclination posterior-mean MAE: {inclination_mae:.3f} rad")

## Output files

The notebook creates:

- `figures/posterior_example.png`
- `figures/parameter_recovery.png`
- `figures/sbc_simple.png`
- `figures/sbc_ecdf.png`
- `figures/sbc_ecdf_difference.png`
- `figures/posterior_contraction.png`
- `figures/inclination_recovery.png`